# How a Computer Learns a Curve: Gradient Descent from Scratch
**Series A: Machine Learning Basics for Chemists, Post 1** &nbsp;|&nbsp; companion notebook

We teach a computer to approximate $y=\sin(x)$ with a cubic polynomial, using only sample points.
Every number quoted in the post is reproduced here.

**How to use this notebook**
1. Run the cells top to bottom (**Runtime → Run all**, or Shift+Enter one cell at a time).
2. Before running a cell marked **Stop and predict**, guess the output.
3. Finish with the **Your turn** exercises at the end.

Nothing needs installing: Colab already has NumPy, Matplotlib and PyTorch.

## 0. Setup
Fix the random seed so everybody gets the same starting point as the post, and build the data.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

SEED = 2026
rng = np.random.default_rng(SEED)

N = 2000                 # number of data points
DEGREE = 3               # cubic: four coefficients
LAM = 1e-3               # ridge penalty strength (lambda)

x = np.linspace(-np.pi, np.pi, N)
y = np.sin(x)            # the target we want the model to learn
z = x / np.pi            # rescaled input, between -1 and 1

# Design matrix: each row is [1, z, z^2, z^3]
X = np.polynomial.polynomial.polyvander(z, DEGREE)
theta0 = rng.standard_normal(DEGREE + 1)   # random starting knobs

print("X shape:", X.shape, "| y shape:", y.shape)
print("first row of X:", X[0])
print("starting theta:", np.round(theta0, 4))

## 1. Features, the intercept, and one prediction by hand
**Stop and predict:** for $z=2/3$ and $\boldsymbol\theta=(1,2,0,-1)$, the post says $\hat y\approx2.037$. What does the feature row look like?

In [ ]:
z_ex = 2/3
row = np.array([1, z_ex, z_ex**2, z_ex**3])
theta_ex = np.array([1, 2, 0, -1])
print("feature row:", np.round(row, 4))
print("prediction :", row @ theta_ex)          # should be about 2.037

# What does the leading 1 do? Change only theta_0 and look at z = 0:
row0 = np.array([1, 0, 0, 0])
for t0 in (0.0, 1.0, 5.0):
    print(f"theta_0 = {t0}: prediction at z=0 is", row0 @ np.array([t0, 2, 0, -1]))

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(x, y, lw=2, label="target: sin(x)")
plt.plot(x, X @ theta0, "--", label="untrained cubic (random knobs)")
plt.xlabel("x"); plt.ylabel("y"); plt.grid(alpha=0.3); plt.legend()
plt.title("Before learning"); plt.show()

## 2. The loss: one number that says how wrong we are
$$L(\boldsymbol\theta)=\frac1N\lVert X\boldsymbol\theta-\mathbf y\rVert^2+\lambda\lVert\boldsymbol\theta\rVert^2$$
The post quotes an initial loss of about **2.79**.

In [ ]:
def loss(theta):
    r = X @ theta - y                       # residuals
    return float(np.mean(r**2) + LAM * np.sum(theta**2))

print("initial loss:", round(loss(theta0), 4))

## 3. Follow the slope downhill
First the one-knob toy problem $L(w)=(w-2)^2$. **Stop and predict:** after one step with $\eta=0.1$ from $w=0$, where is $w$? (The post says 0.4.)

In [ ]:
w, eta = 0.0, 0.1
history = []
for k in range(12):
    history.append((w - 2) ** 2)
    w = w - eta * 2 * (w - 2)          # w <- w - eta * dL/dw
    if k == 0:
        print("after one step: w =", w)
print("after 12 steps: w =", round(w, 4), "(target is 2)")

plt.figure(figsize=(6, 3.5))
plt.semilogy(history, "o-"); plt.xlabel("update"); plt.ylabel("loss")
plt.title("One-knob gradient descent"); plt.grid(alpha=0.3); plt.show()

## 4. Don't trust a gradient you haven't tested
Compare our formula $\nabla L=\frac2N X^{\mathsf T}(X\boldsymbol\theta-\mathbf y)+2\lambda\boldsymbol\theta$ with a central finite difference.

In [ ]:
def gradient(theta):
    r = X @ theta - y
    return (2.0 / N) * (X.T @ r) + 2.0 * LAM * theta

def finite_diff(f, theta, h=1e-6):
    g = np.zeros_like(theta)
    for j in range(theta.size):
        e = np.zeros_like(theta); e[j] = h
        g[j] = (f(theta + e) - f(theta - e)) / (2 * h)
    return g

g_formula = gradient(theta0)
g_numeric = finite_diff(loss, theta0)
print("formula      :", np.round(g_formula, 6))
print("finite diff  :", np.round(g_numeric, 6))
print("max |difference| =", np.max(np.abs(g_formula - g_numeric)))
assert np.allclose(g_formula, g_numeric, rtol=1e-6, atol=1e-8)

## 5. Train with gradient descent
Precompute $H=X^{\mathsf T}X/N+\lambda I$ and $b=X^{\mathsf T}\mathbf y/N$ so that $\nabla L=2(H\boldsymbol\theta-b)$.
We pick the learning rate $\eta_0=1/\mu_{\max}$, where $\mu_{\max}$ is the largest eigenvalue of the Hessian $2H$ (see the post's math supplement).

In [ ]:
H = X.T @ X / N + LAM * np.eye(DEGREE + 1)
b = X.T @ y / N
mu = np.linalg.eigvalsh(2 * H)           # Hessian eigenvalues
eta0 = 1.0 / mu.max()

print("Hessian eigenvalues:", np.round(mu, 4))
print("condition number kappa = %.1f" % (mu.max() / mu.min()))
print("baseline learning rate eta0 = %.4f" % eta0)

# the exact answer key (see section 6), used here only to measure progress
theta_star = np.linalg.solve(H, b)
L_star = loss(theta_star)

def train(theta_start, rate, steps, record=()):
    theta = theta_start.copy()
    losses = [loss(theta)]
    snapshots = {}
    for k in range(1, steps + 1):
        theta = theta - rate * 2.0 * (H @ theta - b)
        losses.append(loss(theta))
        if k in record:
            snapshots[k] = (losses[-1] - L_star, np.linalg.norm(theta - theta_star))
    return theta, np.array(losses), snapshots

theta_gd, losses_gd, snaps = train(theta0, eta0, 2500, record=(10, 100, 500, 1000))
print()
print(f"{'steps':>6} {'loss above optimum':>20} {'distance to theta*':>20}")
for k, (dl, dist) in snaps.items():
    print(f"{k:>6} {dl:>20.3e} {dist:>20.3e}")

In [ ]:
excess = np.maximum(losses_gd - L_star, 1e-16)
plt.figure(figsize=(7, 4))
plt.semilogy(excess); plt.xlabel("step"); plt.ylabel("loss above optimum")
plt.title("Learning curve (log scale)"); plt.grid(alpha=0.3); plt.show()

## 6. The exact answer, and what the fitted curve means
The model is linear in $\boldsymbol\theta$ and the loss is quadratic, so $H\boldsymbol\theta^*=b$ can be solved directly.
**Stop and predict:** which two coefficients will be (nearly) zero, and why?

In [ ]:
print("theta* =", np.round(theta_star, 4))
print("loss at theta* = %.5f" % L_star)
print("gradient descent reached theta* ? ", np.allclose(theta_gd, theta_star, atol=1e-9))

# back to the original variable x:  coefficient of x^j is theta_j / pi^j
c = theta_star / np.pi ** np.arange(DEGREE + 1)
print()
print("sin(x) ~ %.4f x  %+.4f x^3" % (c[1], c[3]))
print("Taylor series: 1.0000 x  -0.1667 x^3")

plt.figure(figsize=(8, 4))
plt.plot(x, y, lw=2, label="sin(x)")
plt.plot(x, X @ theta_star, "--", label="best cubic")
plt.plot(x, x - x**3 / 6, ":", label="Taylor cubic")
plt.ylim(-1.6, 1.6); plt.legend(); plt.grid(alpha=0.3)
plt.xlabel("x"); plt.title("Fitted cubic vs Taylor"); plt.show()

## 7. The learning rate is a dial
Same start, three rates, 150 steps. **Stop and predict:** which one diverges, and at what multiple of $\eta_0$ does the theory say the boundary lies?

In [ ]:
plt.figure(figsize=(7, 4))
print(f"{'rate':>10} {'loss above optimum after 150 steps':>38}")
for m in (0.1, 1.0, 2.2):
    _, ls, _ = train(theta0, m * eta0, 150)
    plt.semilogy(np.maximum(ls - L_star, 1e-16), label=f"{m} x eta0")
    print(f"{m:>7}x   {ls[-1] - L_star:>38.3e}")
plt.xlabel("step"); plt.ylabel("loss above optimum"); plt.legend()
plt.grid(alpha=0.3); plt.title("Effect of the learning rate"); plt.show()
print()
print("Theory: stable only for eta < 2/mu_max, i.e. below 2.0 x eta0")

## 8. Let PyTorch compute the gradient
Autograd applies the chain rule for us. First we confirm that it agrees with our formula; then we train with SGD and Adam.
(If PyTorch is missing, this cell just says so and the rest of the notebook still works.)

In [ ]:
try:
    import torch
    HAVE_TORCH = True
except ImportError:
    HAVE_TORCH = False
    print("PyTorch not installed; skip this section.")

if HAVE_TORCH:
    DT = torch.float64
    Xt = torch.as_tensor(X, dtype=DT)
    yt = torch.as_tensor(y, dtype=DT)

    def loss_t(theta):
        r = Xt @ theta - yt
        return torch.mean(r ** 2) + LAM * torch.sum(theta ** 2)

    # 1) gradient check: autograd vs our formula
    th = torch.tensor(theta0, dtype=DT, requires_grad=True)
    loss_t(th).backward()
    diff = np.max(np.abs(th.grad.numpy() - gradient(theta0)))
    print("autograd vs formula, max |difference| = %.2e" % diff)

    # 2) training loop
    def train_torch(name, steps, lr):
        theta = torch.nn.Parameter(torch.tensor(theta0, dtype=DT))
        opt = (torch.optim.SGD if name == "SGD" else torch.optim.Adam)([theta], lr=lr)
        for _ in range(steps):
            opt.zero_grad(set_to_none=True)   # clear old gradients
            loss_t(theta).backward()          # fill theta.grad
            opt.step()                        # update theta
        return theta.detach().numpy().copy()

    for name, steps, lr in [("SGD", 2500, eta0), ("Adam", 1200, 0.05)]:
        t = train_torch(name, steps, lr)
        print(f"PyTorch {name:<5}: loss above optimum = {loss(t) - L_star:.2e}, "
              f"distance to theta* = {np.linalg.norm(t - theta_star):.2e}")

## 9. Your turn
These match the "Check yourself" exercises in the post.

**Exercise 1.** For $L(w)=(w-2)^2$ with $\eta=0.1$, the error shrinks as $0.8^k$. How many steps until it is below 1% of its start? Verify with the code below.

**Exercise 2.** Which $\eta$ make the toy problem converge? Try $\eta=1.0$ and $\eta=1.1$.

**Exercise 3.** Make `h` in the finite-difference check tiny (`1e-12`). Does the agreement improve or get worse? Why?

**Exercise 4.** Rebuild the design matrix with $x$ instead of $z=x/\pi$. Compute the Hessian eigenvalues and the new condition number. How many steps does gradient descent need now?

In [ ]:
# Exercise 1 and 2: edit eta and rerun
eta = 0.1
w, k = 0.0, 0
while abs(w - 2) > 0.01 * 2 and k < 10_000:
    w = w - eta * 2 * (w - 2)
    k += 1
print("steps until error < 1% of start:", k)

In [ ]:
# Exercise 4 starter: features in the ORIGINAL variable x (no rescaling)
X_raw = np.polynomial.polynomial.polyvander(x, DEGREE)     # [1, x, x^2, x^3]
H_raw = X_raw.T @ X_raw / N + LAM * np.eye(DEGREE + 1)
mu_raw = np.linalg.eigvalsh(2 * H_raw)
print("eigenvalues:", mu_raw)
print("condition number:", mu_raw.max() / mu_raw.min())
# TODO: run gradient descent with eta = 1/mu_raw.max() and count the steps needed
#       to bring the loss within 1e-8 of its minimum. Compare with the rescaled problem.

## 10. Self-check
If everything above ran, these assertions confirm that you reproduced the post's numbers.

In [ ]:
assert abs(loss(theta0) - 2.79) < 0.01
assert abs(eta0 - 0.446) < 0.001
assert abs(L_star - 0.01917) < 1e-4
assert np.allclose(theta_star, [0, 2.572, 0, -2.708], atol=2e-3)
assert snaps[100][0] < 0.02 and snaps[500][0] < 1e-6
print("All checks passed: you reproduced the numbers in the post.")